In [7]:
import re
import ast
import pandas as pd

In [2]:
!ls

gfmag_collect_urls.py  ickg_llm.py	  local_llm.py	__pycache__
gfmag_get_text.py      llm_extraction.py  parser.py	testing.ipynb


In [28]:
test_art = '''=== ARTICLE 118 ===
DATE: 2020-09-11T00:00:00+00:00
URL: https://gfmag.com/economics-policy-regulation/quintessential-capital-management-gabriel-grego/
TRIPLETS_RAW:
[['Quintessential Capital Management:financial_institution', 'has_positive_impact','short activism:event', 'Finance'], ['Gabriel Grego:person', 'is_member_of', 'Quintessential Capital Management:financial_institution', 'Finance'], ['Quintessential Capital Management:financial_institution','relates_to', 'company fraud:event', 'Finance'], ['Quintessential Capital Management:financial_institution','relates_to','management team breaking the law:event', 'Finance'], ['Quintessential Capital Management:financial_institution','relates_to','serious crimes:event', 'Finance'], ['Quintessential Capital Management:financial_institution','relates_to','suspicion of misbehavior:event', 'Finance'], ['Quintessential Capital Management:financial_institution','relates_to','regulators and legal authorities:event', 'Finance'], ['Quintessential Capital Management:financial_institution','relates_to', 'bad ap'''

In [31]:
test_art_clean = test_art

In [62]:
date = re.search(r"DATE:\s*(.+)", test_art)
date = date.group(1).strip() if date else None

url = re.search(r"URL:\s*(.+?)(?=TRIPLETS_RAW|\n|$)", test_art, flags=re.S)
url = url.group(1).strip() if url else None

# including possible truncation
tr = re.search(r"TRIPLETS_RAW:\s*(\[.*)", test_art, flags=re.S)
raw_triplets = tr.group(1) if tr else ""

In [67]:
# ensure it starts with '['
if not raw_triplets.startswith("["):
    raw_triplets = "[" + raw_triplets

In [73]:
# cut off incomplete tail at last ']'
last_bracket = raw_triplets.rfind("]")
raw_triplets = raw_triplets[: last_bracket + 1] + "]"

In [74]:
raw_triplets

"[['Quintessential Capital Management:financial_institution', 'has_positive_impact','short activism:event', 'Finance'], ['Gabriel Grego:person', 'is_member_of', 'Quintessential Capital Management:financial_institution', 'Finance'], ['Quintessential Capital Management:financial_institution','relates_to', 'company fraud:event', 'Finance'], ['Quintessential Capital Management:financial_institution','relates_to','management team breaking the law:event', 'Finance'], ['Quintessential Capital Management:financial_institution','relates_to','serious crimes:event', 'Finance'], ['Quintessential Capital Management:financial_institution','relates_to','suspicion of misbehavior:event', 'Finance'], ['Quintessential Capital Management:financial_institution','relates_to','regulators and legal authorities:event', 'Finance']]"

In [76]:
test_art_clean = {"DATE":date, "URL":url, "TRIPLETS":raw_triplets}

In [77]:
test_art_clean

{'DATE': '2020-09-11T00:00:00+00:00',
 'URL': 'https://gfmag.com/economics-policy-regulation/quintessential-capital-management-gabriel-grego/',
 'TRIPLETS': "[['Quintessential Capital Management:financial_institution', 'has_positive_impact','short activism:event', 'Finance'], ['Gabriel Grego:person', 'is_member_of', 'Quintessential Capital Management:financial_institution', 'Finance'], ['Quintessential Capital Management:financial_institution','relates_to', 'company fraud:event', 'Finance'], ['Quintessential Capital Management:financial_institution','relates_to','management team breaking the law:event', 'Finance'], ['Quintessential Capital Management:financial_institution','relates_to','serious crimes:event', 'Finance'], ['Quintessential Capital Management:financial_institution','relates_to','suspicion of misbehavior:event', 'Finance'], ['Quintessential Capital Management:financial_institution','relates_to','regulators and legal authorities:event', 'Finance']]"}

In [80]:
with open("../data/sustainable_kg_triplets_1000_raw.txt", "r", encoding="latin-1") as f:
    text = f.read(10000)

In [85]:
articles = re.split(r"(?=== ARTICLE \d+ ===)", text)[1:]

In [89]:
articles[1:2]

["== ARTICLE 1 ===\nDATE: 2025-11-07T10:16:56+00:00\nURL: https://gfmag.com/executive-interviews/biat-ceo-jebir-sees-brighter-horizons-for-tunisias-banks/\nTRIPLETS_RAW:\n[['Elyes Jebir:person', 'is_member_of', 'Banque Internationale Arabe De Tunisie:company', 'Financial Institution'], ['Banque Internationale Arabe De Tunisie:company', 'has_positive_impact', 'Tunisian banking sector:sector', 'Financial Institution'], ['Tunisia:country', 'has_exposure', 'Covid-19 pandemic:event', 'Economic Indicator'], ['Tunisia:country', 'has_exposure', 'War in Ukraine:event', 'Economic Indicator'], ['Tunisia:country', 'has_exposure', 'Middle East developments:event', 'Economic Indicator'], ['Tunisia:country', 'has_exposure', 'International context:event', 'Economic Indicator'], ['Tunisia:country', 'has_exposure', 'New legislation:event', 'Economic Indicator'], ['Tunisia:country', 'has_exposure', 'Instant payments:financial_instrument', '\n\n="]

In [102]:
def parse_text(path:str):
    """ Parse ARTICLE blocks into clean dictionaries and write them into a dataframe"""
    rows = []

    with open(path, "r", encoding="latin-1") as f:
        text = f.read()
        
    articles = re.split(r"(?=== ARTICLE \d+ ===)", text)[1:]
    
    for art in articles: 
        date = re.search(r"DATE:\s*(.+)", art)
        date = date.group(1).strip() if date else None
        
        url = re.search(r"URL:\s*(.+?)(?=TRIPLETS_RAW|\n|$)", art, flags=re.S)
        url = url.group(1).strip() if url else None
        
        # including possible truncation
        tr = re.search(r"TRIPLETS_RAW:\s*(\[.*)", art, flags=re.S)
        raw_triplets = tr.group(1) if tr else ""
    
        # ensure it starts with '['
        if not raw_triplets.startswith("["):
            raw_triplets = "[" + raw_triplets
            
        # cut off incomplete tail at last ']'
        last_bracket = raw_triplets.rfind("]")
        raw_triplets = raw_triplets[: last_bracket + 1] + "]"
    
        rows.append({"DATE": date, "URL": url, "TRIPLETS": raw_triplets})

    return pd.DataFrame(rows)


In [103]:
parse_text(path="../data/sustainable_kg_triplets_1000_raw.txt")

,DATE,URL,TRIPLETS
0,2025-11-09T09:30:00+00:00,https://gfmag.com/sustainable-finance/coal-tak...,"[['Renewable energy:economic_indicator', 'is_c..."
1,2025-11-07T10:16:56+00:00,https://gfmag.com/executive-interviews/biat-ce...,"[['Elyes Jebir:person', 'is_member_of', 'Banqu..."
2,2025-11-04T16:10:02+00:00,https://gfmag.com/sustainable-finance/cop-30-m...,[['UN Framework Convention on Climate Change:r...
3,2025-11-20T15:49:52+00:00,https://gfmag.com/sustainable-finance/garanti-...,"[['Garanti BBVA:company', 'has_positive_impact..."
4,2025-05-21T10:13:14+00:00,https://gfmag.com/sustainable-finance/powering...,"[['Project finance:financial_instrument', 'has..."
...,...,...,...
114,2020-11-13T00:00:00+00:00,https://gfmag.com/sustainable-finance/green-bo...,"[['Green bonds:financial_instrument', 'has_pos..."
115,2020-11-11T00:00:00+00:00,https://gfmag.com/sustainable-finance/r-edward...,"[['R. Edward Freeman:person','relates_to','sta..."
116,2020-11-11T00:00:00+00:00,https://gfmag.com/features/sustainable-corpora...,"[['NRG Energy:company', 'has_positive_impact',..."
117,2020-10-10T00:00:00+00:00,https://gfmag.com/sustainable-finance/citi-bre...,"[['Jane Fraser:person', 'is_member_of', 'Citig..."


# functions

In [ ]:
# TODO: category, object types

In [ ]:
def parse_text(path:str):
    """ Parse ARTICLE blocks into clean dictionaries and write them into a dataframe"""
    rows = []

    with open(path, "r", encoding="latin-1") as f:
        text = f.read()
        
    articles = re.split(r"(?=== ARTICLE \d+ ===)", text)[1:]
    
    for art in articles: 
        date = re.search(r"DATE:\s*(.+)", art)
        date = date.group(1).strip() if date else None
        
        url = re.search(r"URL:\s*(.+?)(?=TRIPLETS_RAW|\n|$)", art, flags=re.S)
        url = url.group(1).strip() if url else None
        
        # including possible truncation
        tr = re.search(r"TRIPLETS_RAW:\s*(\[.*)", art, flags=re.S)
        raw_triplets = tr.group(1) if tr else ""
    
        # ensure it starts with '['
        if not raw_triplets.startswith("["):
            raw_triplets = "[" + raw_triplets
            
        # cut off incomplete tail at last ']'
        last_bracket = raw_triplets.rfind("]")
        raw_triplets = raw_triplets[: last_bracket + 1] + "]"
    
        rows.append({"DATE": date, "URL": url, "TRIPLETS": raw_triplets})

    return pd.DataFrame(rows)
